# Human Validation Analysis (9 Files)

This notebook extends the 3-way agreement workflow to **9 input files**:
- 3 files from Human A
- 3 files from Human B
- 3 files from LLM

It computes:
1. Human A vs Human B
2. Human A vs LLM
3. Human B vs LLM
4. Average of (Human A vs LLM) and (Human B vs LLM)

Before final scoring, it also reports where KC tags differ across Human A, Human B, and LLM.

In [ ]:
import json
import os
from pathlib import Path
from typing import Dict, List, Set, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# -----------------------------
# Configure your 9 input files
# -----------------------------
HUMAN_A_FILES = [
    "dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_10155_1774736175604.json",
    "dataset/Rater_KC_Tags/REPLACE_WITH_HUMAN_A_FILE_2.json",
    "dataset/Rater_KC_Tags/REPLACE_WITH_HUMAN_A_FILE_3.json",
]

HUMAN_B_FILES = [
    "dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_10155_1774820622134.json",
    "dataset/Rater_KC_Tags/REPLACE_WITH_HUMAN_B_FILE_2.json",
    "dataset/Rater_KC_Tags/REPLACE_WITH_HUMAN_B_FILE_3.json",
]

LLM_FILES = [
    "results/human_validation/llm_annotations_10155.json",
    "results/human_validation/REPLACE_WITH_LLM_FILE_2.json",
    "results/human_validation/REPLACE_WITH_LLM_FILE_3.json",
]

ALL_KCS = [
    "If/Else", "NestedIf", "While", "For", "NestedFor",
    "Math+-*/", "Math%", "LogicAndNotOr", "LogicCompareNum",
    "LogicBoolean", "StringFormat", "StringConcat", "StringIndex",
    "StringLen", "StringEqual", "CharEqual", "ArrayIndex", "DefFunction"
]

print(f"Working directory: {os.getcwd()}")

In [ ]:
def load_annotations(filepath: str) -> Tuple[str, Dict[str, Set[str]]]:
    """
    Load annotations from one JSON file.
    Expected format:
      {
        "rater": "name",
        "annotations": {
          "problem_id": {"gaps": ["KC1", ...]} OR ["KC1", ...] OR null
        }
      }
    """
    path = Path(filepath)
    if not path.exists():
        print(f"MISSING: {filepath}")
        return path.stem, {}

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    rater_name = data.get("rater", path.stem)
    parsed: Dict[str, Set[str]] = {}
    raw_annotations = data.get("annotations", {})

    for pid, val in raw_annotations.items():
        if isinstance(val, dict) and "gaps" in val:
            gaps = val.get("gaps")
            parsed[pid] = set(gaps) if isinstance(gaps, list) else set()
        elif isinstance(val, list):
            parsed[pid] = set(val)
        else:
            parsed[pid] = set()

    print(f"Loaded: {filepath} ({len(parsed)} problems)")
    return rater_name, parsed


def merge_rater_files(filepaths: List[str], group_label: str) -> Tuple[str, Dict[str, Set[str]]]:
    """
    Merge multiple files for one rater group.
    If the same problem ID appears in multiple files, KCs are unioned.
    """
    merged: Dict[str, Set[str]] = {}
    observed_names: List[str] = []
    duplicate_problem_hits = 0

    for fp in filepaths:
        name, anns = load_annotations(fp)
        observed_names.append(name)

        for pid, gaps in anns.items():
            if pid in merged:
                duplicate_problem_hits += 1
                merged[pid] = merged[pid].union(gaps)
            else:
                merged[pid] = set(gaps)

    canonical_name = observed_names[0] if observed_names else group_label
    print("")
    print(f"[{group_label}] merged problems: {len(merged)}")
    print(f"[{group_label}] duplicate problem merges: {duplicate_problem_hits}")
    print(f"[{group_label}] source raters seen: {sorted(set(observed_names))}")
    return canonical_name, merged

In [ ]:
def compute_metrics(name_a: str, anns_a: Dict[str, Set[str]], name_b: str, anns_b: Dict[str, Set[str]], common_pids: List[str]):
    """Compute Cohen's kappa, F1, precision, recall and supporting stats."""
    y_a = []
    y_b = []

    for pid in common_pids:
        gaps_a = anns_a.get(pid, set())
        gaps_b = anns_b.get(pid, set())
        for kc in ALL_KCS:
            y_a.append(1 if kc in gaps_a else 0)
            y_b.append(1 if kc in gaps_b else 0)

    y_a = np.array(y_a)
    y_b = np.array(y_b)
    n = len(y_a)

    if n == 0:
        return {
            "name_a": name_a, "name_b": name_b, "kappa": 0.0, "f1": 0.0,
            "precision": 0.0, "recall": 0.0, "observed_agreement": 0.0,
            "tp": 0, "fp": 0, "fn": 0, "n_problems": 0, "kc_kappas": {}
        }

    po = np.sum(y_a == y_b) / n
    pe = (np.sum(y_a == 1) * np.sum(y_b == 1) + np.sum(y_a == 0) * np.sum(y_b == 0)) / (n * n)
    kappa = (po - pe) / (1 - pe) if (1 - pe) > 1e-12 else 0.0

    tp = int(np.sum((y_a == 1) & (y_b == 1)))
    fp = int(np.sum((y_a == 0) & (y_b == 1)))
    fn = int(np.sum((y_a == 1) & (y_b == 0)))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    kc_kappas = {}
    n_kc = len(common_pids)

    for kc in ALL_KCS:
        yh = [1 if kc in anns_a.get(pid, set()) else 0 for pid in common_pids]
        yl = [1 if kc in anns_b.get(pid, set()) else 0 for pid in common_pids]

        if n_kc == 0:
            kc_kappas[kc] = {"kappa": 0.0, "count_a": 0, "count_b": 0, "both": 0}
            continue

        po_kc = sum(1 for a, b in zip(yh, yl) if a == b) / n_kc
        pe_kc = (sum(yh) * sum(yl) + (n_kc - sum(yh)) * (n_kc - sum(yl))) / (n_kc * n_kc)
        k = (po_kc - pe_kc) / (1 - pe_kc) if (1 - pe_kc) > 1e-12 else 0.0

        kc_kappas[kc] = {
            "kappa": k,
            "count_a": int(sum(yh)),
            "count_b": int(sum(yl)),
            "both": int(sum(1 for a, b in zip(yh, yl) if a == 1 and b == 1))
        }

    return {
        "name_a": name_a,
        "name_b": name_b,
        "kappa": float(kappa),
        "observed_agreement": float(po),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "kc_kappas": kc_kappas,
        "n_problems": len(common_pids)
    }


def interpret_kappa(k: float) -> str:
    if k >= 0.8:
        return "Almost perfect"
    if k >= 0.6:
        return "Substantial"
    if k >= 0.4:
        return "Moderate"
    if k >= 0.2:
        return "Fair"
    if k >= 0.0:
        return "Slight"
    return "Poor"


def print_pairwise_result(r: dict):
    print(f"\n{r['name_a']} vs {r['name_b']}")
    print("-" * 64)
    print(f"Kappa:              {r['kappa']:.3f} ({interpret_kappa(r['kappa'])})")
    print(f"Observed agreement: {r['observed_agreement']:.3f}")
    print(f"F1:                 {r['f1']:.3f}")
    print(f"Precision:          {r['precision']:.3f}")
    print(f"Recall:             {r['recall']:.3f}")
    print(f"TP/FP/FN:           {r['tp']}/{r['fp']}/{r['fn']}")
    print(f"Common problems:    {r['n_problems']}")

In [ ]:
def kc_difference_report(anns_a: Dict[str, Set[str]], anns_b: Dict[str, Set[str]], anns_l: Dict[str, Set[str]], common_pids: List[str]):
    """
    Report KC-level disagreement patterns among Human A, Human B, and LLM
    before final aggregate scoring.
    """
    rows = []

    for kc in ALL_KCS:
        stats = {
            "A_only": 0,
            "B_only": 0,
            "LLM_only": 0,
            "A_B_not_LLM": 0,
            "A_LLM_not_B": 0,
            "B_LLM_not_A": 0,
            "all_equal": 0
        }

        for pid in common_pids:
            a = kc in anns_a.get(pid, set())
            b = kc in anns_b.get(pid, set())
            l = kc in anns_l.get(pid, set())

            true_count = int(a) + int(b) + int(l)
            if true_count == 0 or true_count == 3:
                stats["all_equal"] += 1
            elif a and not b and not l:
                stats["A_only"] += 1
            elif b and not a and not l:
                stats["B_only"] += 1
            elif l and not a and not b:
                stats["LLM_only"] += 1
            elif a and b and not l:
                stats["A_B_not_LLM"] += 1
            elif a and l and not b:
                stats["A_LLM_not_B"] += 1
            elif b and l and not a:
                stats["B_LLM_not_A"] += 1

        disagree_count = len(common_pids) - stats["all_equal"]
        rows.append({
            "kc": kc,
            "disagree_count": disagree_count,
            "disagree_rate": (disagree_count / len(common_pids)) if common_pids else 0.0,
            **stats
        })

    rows.sort(key=lambda x: x["disagree_count"], reverse=True)

    print("\n" + "=" * 80)
    print("KC DIFFERENCE REPORT (A vs B vs LLM)")
    print("=" * 80)
    header = (
        f"{'KC':<18} {'Diff#':>6} {'Diff%':>8} {'A_only':>7} {'B_only':>7} {'L_only':>7} "
        f"{'A&B!L':>7} {'A&L!B':>7} {'B&L!A':>7}"
    )
    print(header)
    print("-" * len(header))

    for row in rows:
        print(
            f"{row['kc']:<18} {row['disagree_count']:>6} {row['disagree_rate']*100:>7.1f}% "
            f"{row['A_only']:>7} {row['B_only']:>7} {row['LLM_only']:>7} "
            f"{row['A_B_not_LLM']:>7} {row['A_LLM_not_B']:>7} {row['B_LLM_not_A']:>7}"
        )

    print("\nTop KCs with highest disagreement:")
    for row in rows[:5]:
        print(f"- {row['kc']}: {row['disagree_count']} / {len(common_pids)} ({row['disagree_rate']*100:.1f}%)")

    return rows


def problem_level_difference_examples(anns_a: Dict[str, Set[str]], anns_b: Dict[str, Set[str]], anns_l: Dict[str, Set[str]], common_pids: List[str], max_rows: int = 15):
    """Show problem-level examples where raters disagree on KC tags."""
    examples = []

    for pid in common_pids:
        a = anns_a.get(pid, set())
        b = anns_b.get(pid, set())
        l = anns_l.get(pid, set())

        if a == b == l:
            continue

        examples.append({
            "pid": pid,
            "a_only": sorted(list(a - b - l)),
            "b_only": sorted(list(b - a - l)),
            "llm_only": sorted(list(l - a - b)),
            "a_b_not_llm": sorted(list((a & b) - l)),
            "a_llm_not_b": sorted(list((a & l) - b)),
            "b_llm_not_a": sorted(list((b & l) - a)),
        })

    print("\n" + "=" * 80)
    print("PROBLEM-LEVEL DIFFERENCE EXAMPLES")
    print("=" * 80)
    print(f"Total problems with any disagreement: {len(examples)} / {len(common_pids)}")

    for row in examples[:max_rows]:
        print(f"\nProblem {row['pid']}")
        print(f"  A_only:       {row['a_only']}")
        print(f"  B_only:       {row['b_only']}")
        print(f"  LLM_only:     {row['llm_only']}")
        print(f"  A&B not LLM:  {row['a_b_not_llm']}")
        print(f"  A&LLM not B:  {row['a_llm_not_b']}")
        print(f"  B&LLM not A:  {row['b_llm_not_a']}")

    if len(examples) > max_rows:
        print(f"\n... showing first {max_rows} of {len(examples)} disagreements")

    return examples

In [ ]:
# Merge 3 files per rater group
name_a, anns_a = merge_rater_files(HUMAN_A_FILES, "Human A")
name_b, anns_b = merge_rater_files(HUMAN_B_FILES, "Human B")
name_l, anns_l = merge_rater_files(LLM_FILES, "LLM")

# Intersect only problems present in all three groups
common_pids = sorted(set(anns_a.keys()) & set(anns_b.keys()) & set(anns_l.keys()), key=lambda x: int(x) if str(x).isdigit() else str(x))

print("\n" + "=" * 80)
print("DATA COVERAGE")
print("=" * 80)
print(f"Human A merged problems: {len(anns_a)}")
print(f"Human B merged problems: {len(anns_b)}")
print(f"LLM merged problems:     {len(anns_l)}")
print(f"Common problems (A & B & LLM): {len(common_pids)}")

if len(common_pids) == 0:
    raise ValueError("No common problem IDs across the 3 merged groups. Check file paths and IDs.")

In [ ]:
# Required pre-final analysis: show where KCs differ across A, B, and LLM
kc_diff_rows = kc_difference_report(anns_a, anns_b, anns_l, common_pids)
problem_diff_rows = problem_level_difference_examples(anns_a, anns_b, anns_l, common_pids, max_rows=15)

In [ ]:
# Pairwise comparisons
r_ab = compute_metrics(name_a, anns_a, name_b, anns_b, common_pids)
r_al = compute_metrics(name_a, anns_a, name_l, anns_l, common_pids)
r_bl = compute_metrics(name_b, anns_b, name_l, anns_l, common_pids)

print("\n" + "=" * 80)
print("PAIRWISE AGREEMENT METRICS")
print("=" * 80)
print_pairwise_result(r_ab)
print_pairwise_result(r_al)
print_pairwise_result(r_bl)

avg_vs_llm = {
    "kappa": (r_al["kappa"] + r_bl["kappa"]) / 2,
    "f1": (r_al["f1"] + r_bl["f1"]) / 2,
    "precision": (r_al["precision"] + r_bl["precision"]) / 2,
    "recall": (r_al["recall"] + r_bl["recall"]) / 2
}

human_baseline = r_ab["kappa"]
ratio_vs_human = (avg_vs_llm["kappa"] / human_baseline) if human_baseline > 0 else 0.0

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
print(f"Human A vs Human B kappa (baseline): {human_baseline:.3f}")
print(f"Average (Human A vs LLM, Human B vs LLM) kappa: {avg_vs_llm['kappa']:.3f}")
print(f"Average vs LLM reaches {ratio_vs_human*100:.1f}% of human baseline")

print("\nSummary table:")
print(f"{'Pair':<26} {'Kappa':>8} {'F1':>8} {'Precision':>10} {'Recall':>8}")
print("-" * 68)
print(f"{'Human A vs Human B':<26} {r_ab['kappa']:>8.3f} {r_ab['f1']:>8.3f} {r_ab['precision']:>10.3f} {r_ab['recall']:>8.3f}")
print(f"{'Human A vs LLM':<26} {r_al['kappa']:>8.3f} {r_al['f1']:>8.3f} {r_al['precision']:>10.3f} {r_al['recall']:>8.3f}")
print(f"{'Human B vs LLM':<26} {r_bl['kappa']:>8.3f} {r_bl['f1']:>8.3f} {r_bl['precision']:>10.3f} {r_bl['recall']:>8.3f}")
print(f"{'Average vs LLM':<26} {avg_vs_llm['kappa']:>8.3f} {avg_vs_llm['f1']:>8.3f} {avg_vs_llm['precision']:>10.3f} {avg_vs_llm['recall']:>8.3f}")

In [ ]:
# Visualizations
pair_labels = ["Human A vs Human B", "Human A vs LLM", "Human B vs LLM", "Average vs LLM"]
pair_kappas = [r_ab['kappa'], r_al['kappa'], r_bl['kappa'], avg_vs_llm['kappa']]

plt.figure(figsize=(11, 6))
bar_colors = ["#ffb347", "#aec6cf", "#aec6cf", "#77dd77"]
bars = plt.bar(pair_labels, pair_kappas, color=bar_colors)

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h + 0.01, f"{h:.3f}", ha="center", va="bottom", fontweight="bold")

plt.axhline(y=r_ab['kappa'], color="#d62728", linestyle="--", alpha=0.6, label="Human baseline")
plt.title(f"Agreement Comparison (Average vs LLM = {ratio_vs_human*100:.1f}% of human baseline)")
plt.ylabel("Cohen's kappa")
plt.ylim(0, max(pair_kappas + [0.1]) + 0.1)
plt.grid(axis="y", linestyle=":", alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

names = [name_a, name_b, name_l]
matrix = np.array([
    [1.0, r_ab['kappa'], r_al['kappa']],
    [r_ab['kappa'], 1.0, r_bl['kappa']],
    [r_al['kappa'], r_bl['kappa'], 1.0]
])

plt.figure(figsize=(7, 5))
sns.heatmap(matrix, annot=True, fmt=".3f", xticklabels=names, yticklabels=names, cmap="Blues", vmin=0, vmax=1)
plt.title("Pairwise Cohen kappa Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# Optional: save outputs
output = {
    "n_common_problems": len(common_pids),
    "rater_groups": {
        "human_a_files": HUMAN_A_FILES,
        "human_b_files": HUMAN_B_FILES,
        "llm_files": LLM_FILES
    },
    "pairwise": {
        "human_a_vs_human_b": r_ab,
        "human_a_vs_llm": r_al,
        "human_b_vs_llm": r_bl
    },
    "average_vs_llm": avg_vs_llm,
    "ratio_vs_human_baseline": ratio_vs_human,
    "kc_difference_report": kc_diff_rows,
    "problem_difference_examples": problem_diff_rows
}

output_path = Path("results/human_validation/human_validation_results_9files.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f:
    json.dump(output, f, indent=2)

print(f"Saved results to: {output_path}")